In [ ]:
import pandas as pd
import os
import csv
import re
import numpy as np
import country_converter as coco
import statsmodels.formula.api as smf
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import statsmodels.api as sm

In [ ]:
path = r"C:\Users\psimo\Music\Python\Paper\RU_trade.csv"
df = pd.read_csv(path, nrows=50000)

print(df.shape)
display(df.head())

In [ ]:
# leave real countries: ISO3 code (A-Z, 3 letters)
df = df[df["COUNTERPART_COUNTRY.ID"].astype(str).str.fullmatch(r"[A-Z]{3}")]

print(df.shape)
display(df[["COUNTERPART_COUNTRY.ID","COUNTERPART_COUNTRY","TRADE_FLOW","2016","2022","2024"]].head())

In [ ]:
years = [str(y) for y in range(2016, 2025)]

# leave only those partners, who have at least one data point per period
df = df[df[years].notna().any(axis=1)]

print(df.shape)
display(df[["COUNTERPART_COUNTRY.ID","COUNTERPART_COUNTRY","TRADE_FLOW","2016","2022","2024"]].head(10))

In [ ]:
# ---- 3) make the numeric years and discard rows where there are 0 or NaN ----
df[years] = df[years].apply(pd.to_numeric, errors="coerce")
df = df[df[years].fillna(0).sum(axis=1) > 0]

print("Filtered (rows, cols):", df.shape)

# ---- 4) long format ----
df_long = df.melt(
    id_vars=["COUNTERPART_COUNTRY.ID", "COUNTERPART_COUNTRY", "TRADE_FLOW"],
    value_vars=years,
    var_name="year",
    value_name="value"
)
df_long["year"] = df_long["year"].astype(int)

# ---- 5) imports/exports to separate columns ----
panel = df_long.pivot_table(
    index=["COUNTERPART_COUNTRY.ID", "COUNTERPART_COUNTRY", "year"],
    columns="TRADE_FLOW",
    values="value",
    aggfunc="sum"
).reset_index()

panel.columns.name = None

# ---- 6) discards years where both streams are absent (0/NaN) ----
flow_cols = [c for c in panel.columns if c not in ["COUNTERPART_COUNTRY.ID", "COUNTERPART_COUNTRY", "year"]]
panel[flow_cols] = panel[flow_cols].fillna(0)
panel = panel[panel[flow_cols].sum(axis=1) > 0]

print("Final panel (rows, cols):", panel.shape)
display(panel.head(50))

# (optional) save
panel.to_csv(r"C:\Users\psimo\Music\Python\Paper\RU_trade_clean_panel.csv", index=False)
print("Saved:", r"C:\Users\psimo\Music\Python\Paper\RU_trade_clean_panel.csv")

In [ ]:
partners = (panel[["COUNTERPART_COUNTRY.ID","COUNTERPART_COUNTRY"]]
            .drop_duplicates()
            .sort_values("COUNTERPART_COUNTRY.ID")
            .reset_index(drop=True))
partners.to_csv("partners_from_data.csv", index=False)
partners.head()

In [ ]:
url = "https://raw.githubusercontent.com/world-politics-datalab/un_members_countryname_iso2_iso3_region/main/un_member_iso_region.csv"
un = pd.read_csv(url)

un_iso3 = (un["iso3"].astype(str).str.upper().dropna().drop_duplicates().sort_values())
un_iso3.to_frame("iso3").to_csv("un_members_iso3.csv", index=False)

print("UN ISO3 count:", len(un_iso3))
un_iso3.head()

In [ ]:
# 1) add UN ISO3
un = pd.read_csv("un_members_iso3.csv")
un_set = set(un["iso3"].astype(str).str.upper())

# 2) how many partners before the filter?
before = panel["COUNTERPART_COUNTRY.ID"].nunique()

# 3) filtering
panel_un = panel[panel["COUNTERPART_COUNTRY.ID"].isin(un_set)].copy()

after = panel_un["COUNTERPART_COUNTRY.ID"].nunique()
print("Partners before:", before, "after UN filter:", after)

# 4) what's dropped
dropped = (panel[~panel["COUNTERPART_COUNTRY.ID"].isin(un_set)]
           [["COUNTERPART_COUNTRY.ID","COUNTERPART_COUNTRY"]]
           .drop_duplicates()
           .sort_values("COUNTERPART_COUNTRY.ID"))

dropped.to_csv("dropped_non_UN_partners.csv", index=False)
panel_un.to_csv("RU_trade_panel_UN.csv", index=False)

display(dropped.head(30))
display(panel_un.head(20))

In [ ]:
# UN-only
panel_un = panel[panel["COUNTERPART_COUNTRY.ID"].isin(un_set)].copy()

# UN + (optional) territories (robustness)
extra = {"HKG","TWN"} 
panel_un_plus = panel[panel["COUNTERPART_COUNTRY.ID"].isin(un_set.union(extra))].copy()

In [ ]:
gsdb = pd.read_csv(r"C:\Users\psimo\Music\Python\Paper\GSDB_V4.csv")

# 1) leave only the cases, where target is Russia
gsdb_ru = gsdb[gsdb["sanctioned_state"].astype(str).str.contains("Russia", case=False, na=False)].copy()

# 2) adjust the yers
gsdb_ru["begin"] = pd.to_numeric(gsdb_ru["begin"], errors="coerce")
gsdb_ru["end"]   = pd.to_numeric(gsdb_ru["end"], errors="coerce").fillna(2023)  # GSDB iki 2023

# 3) intensity: sum (0..6)
tools = ["trade","arms","military","financial","travel","other"]
gsdb_ru[tools] = gsdb_ru[tools].apply(pd.to_numeric, errors="coerce").fillna(0)
gsdb_ru["intensity"] = gsdb_ru[tools].sum(axis=1)

# 4)  sanctioning_state, year
rows = []
for _, r in gsdb_ru[["sanctioning_state","begin","end","intensity"]].dropna().iterrows():
    b, e = int(r["begin"]), int(r["end"])
    if e < b:
        continue
    for y in range(b, e+1):
        rows.append((r["sanctioning_state"], y, r["intensity"]))

gsdb_it = pd.DataFrame(rows, columns=["sanctioning_state", "year", "gsdb_intensity"])

# 5) if there are few episodes during the same period – sum
gsdb_it = gsdb_it.groupby(["sanctioning_state","year"], as_index=False)["gsdb_intensity"].sum()

display(gsdb_it.head(10))
print("Rows:", gsdb_it.shape)

In [ ]:
# --- 0) normalization function ---
def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"&", " and ", s)
    s = re.sub(r"[\.,'\(\)]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

# --- 1) partner dictionary from trade panel (UN-only) ---
partner_map = (panel_un[["COUNTERPART_COUNTRY.ID", "COUNTERPART_COUNTRY"]]
               .drop_duplicates()
               .assign(name_norm=lambda d: d["COUNTERPART_COUNTRY"].apply(norm)))

# if accidentally two identical name_norm -> leave the first one (rare, but protection)
partner_map = partner_map.drop_duplicates(subset=["name_norm"])

# --- 2) GSDB: take RU target cases and make intensity as before ---
gsdb = pd.read_csv(r"C:\Users\psimo\Music\Python\Paper\GSDB_V4.csv")
gsdb_ru = gsdb[gsdb["sanctioned_state"].astype(str).str.contains("Russia", case=False, na=False)].copy()

gsdb_ru["begin"] = pd.to_numeric(gsdb_ru["begin"], errors="coerce")
gsdb_ru["end"]   = pd.to_numeric(gsdb_ru["end"], errors="coerce").fillna(2023)

tools = ["trade","arms","military","financial","travel","other"]
gsdb_ru[tools] = gsdb_ru[tools].apply(pd.to_numeric, errors="coerce").fillna(0)
gsdb_ru["intensity"] = gsdb_ru[tools].sum(axis=1)

rows = []
for _, r in gsdb_ru[["sanctioning_state","begin","end","intensity"]].dropna().iterrows():
    b, e = int(r["begin"]), int(r["end"])
    if e < b:
        continue
    for y in range(b, e+1):
        rows.append((r["sanctioning_state"], y, r["intensity"]))

gsdb_it = pd.DataFrame(rows, columns=["sanctioning_state","year","gsdb_intensity"])
gsdb_it = gsdb_it.groupby(["sanctioning_state","year"], as_index=False)["gsdb_intensity"].sum()

# --- 3) minimal fixes to GSDB names (only a few common ones) ---
fix_names = {
    "united states": "united states of america",
    "russia": "russian federation",
    "iran": "iran islamic republic of",
    "syria": "syrian arab republic",
    "korea, south": "korea republic of",
    "korea, north": "korea democratic peoples republic of",
    "czech republic": "czechia",
    "slovak republic": "slovakia",
    "venezuela": "venezuela bolivarian republic of",
    # EU/organizations – leave as non-match (they will naturally drop out)
    "eu": None,
    "european union": None,
    "un": None,
    "united nations": None,
}

gsdb_it["name_norm"] = gsdb_it["sanctioning_state"].apply(norm)
gsdb_it["name_norm"] = gsdb_it["name_norm"].map(lambda x: fix_names.get(x, x))
gsdb_it = gsdb_it[gsdb_it["name_norm"].notna()]  # drop EU/UN etc.

# --- 4) connect: GSDB sender name -> panel partner ISO3 ---
gsdb_it2 = gsdb_it.merge(partner_map[["name_norm","COUNTERPART_COUNTRY.ID"]],
                         on="name_norm", how="left")


unmatched = (gsdb_it2[gsdb_it2["COUNTERPART_COUNTRY.ID"].isna()]["sanctioning_state"]
             .drop_duplicates().sort_values())
print("Unmatched sanctioning_state:", len(unmatched))
display(unmatched.head(50))

# --- 5) leave only the ones that are mapped and connect to the trade panel ---
gsdb_ready = gsdb_it2.dropna(subset=["COUNTERPART_COUNTRY.ID"]).copy()

panel_g = panel_un.merge(gsdb_ready[["COUNTERPART_COUNTRY.ID","year","gsdb_intensity"]],
                         on=["COUNTERPART_COUNTRY.ID","year"], how="left")
panel_g["gsdb_intensity"] = panel_g["gsdb_intensity"].fillna(0)

display(panel_g.head(20))

In [ ]:
EU27 = {
 "AUT","BEL","BGR","HRV","CYP","CZE","DNK","EST","FIN","FRA","DEU","GRC","HUN","IRL","ITA",
 "LVA","LTU","LUX","MLT","NLD","POL","PRT","ROU","SVK","SVN","ESP","SWE"
}
G7 = {"CAN","FRA","DEU","ITA","JPN","GBR","USA"}

In [ ]:
def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"&", " and ", s)
    s = re.sub(r"[\.,'\(\)]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

# dictionary from panel (UN-only): partner name -> ISO3
partner_map = (panel_un[["COUNTERPART_COUNTRY.ID","COUNTERPART_COUNTRY"]]
               .drop_duplicates()
               .assign(name_norm=lambda d: d["COUNTERPART_COUNTRY"].apply(norm))
               .drop_duplicates(subset=["name_norm"]))
name_to_iso3 = dict(zip(partner_map["name_norm"], partner_map["COUNTERPART_COUNTRY.ID"]))

# alias fixes for single names
alias = {
    "united states": "usa",          # with the purpose
    "usa": "united states of america",
    "czech republic": "czechia",
    "korea south": "korea republic of",
    "korea, south": "korea republic of",
    "taiwan": None,                  # UN-only: drop (TWN is not UN member)
    "liechtenstein": "liechtenstein",
    "monaco": "monaco",
    "poland": "poland",
    "kazakhstan": "kazakhstan",
    "macedonia": "north macedonia",
}

def to_iso3(token):
    t = norm(token)
    t = alias.get(t, t)
    if t is None:
        return None
    return name_to_iso3.get(t)

# fetch unmapped rows from gsdb_it (17 names)
unmatched_names = set(unmatched.astype(str).tolist())
extra_rows = []

for _, r in gsdb_it[gsdb_it["sanctioning_state"].isin(unmatched_names)].iterrows():
    name = str(r["sanctioning_state"])
    y = int(r["year"])
    inten = float(r["gsdb_intensity"])

    parts = [p.strip() for p in name.split(",")]

    iso_list = set()
    # if there is EU -> add EU27
    if any(p.strip().upper() == "EU" for p in parts) or "European Union" in name:
        iso_list |= EU27
    # if there is G7 -> add G7
    if any(p.strip().upper() == "G7" for p in parts):
        iso_list |= G7

    # add individual countries if they pull out
    for p in parts:
        if p.strip().upper() in {"EU","G7"}:
            continue
        iso = to_iso3(p)
        if iso:
            iso_list.add(iso)

    # create raws
    for iso in iso_list:
        extra_rows.append((iso, y, inten))

extra = pd.DataFrame(extra_rows, columns=["COUNTERPART_COUNTRY.ID","year","gsdb_intensity"])
extra = extra.groupby(["COUNTERPART_COUNTRY.ID","year"], as_index=False)["gsdb_intensity"].sum()

print("Extra rows created:", extra.shape)
display(extra.head(20))

In [ ]:
gsdb_ready = gsdb_it2.dropna(subset=["COUNTERPART_COUNTRY.ID"])[["COUNTERPART_COUNTRY.ID","year","gsdb_intensity"]].copy()

gsdb_ready2 = pd.concat([gsdb_ready, extra], ignore_index=True)
# 1) reaggregate with MAX
gsdb_ready2 = gsdb_ready2.groupby(["COUNTERPART_COUNTRY.ID","year"], as_index=False)["gsdb_intensity"].max()

# 2) merge with trade
panel_g2 = panel_un.merge(gsdb_ready2, on=["COUNTERPART_COUNTRY.ID","year"], how="left")
panel_g2["gsdb_intensity"] = panel_g2["gsdb_intensity"].fillna(0)
display(panel_g2.head(20))

In [ ]:
(panel_g2["gsdb_intensity"] > 0).mean()

(panel_g2.groupby("COUNTERPART_COUNTRY.ID")["gsdb_intensity"].sum()
 .sort_values(ascending=False).head(15))

In [ ]:
#eg. Australia
panel_g2.loc[panel_g2["COUNTERPART_COUNTRY.ID"]=="AUS", ["year","gsdb_intensity"]].sort_values("year")

In [ ]:
panel_g2["gsdb_intensity"].max()

In [ ]:
panel_g2.loc[panel_g2["gsdb_intensity"] == panel_g2["gsdb_intensity"].max(),
             ["COUNTERPART_COUNTRY.ID","COUNTERPART_COUNTRY","year","gsdb_intensity"]].head(20)

In [ ]:
# 0 = none, 6 = all tool categories active
panel_g2["gsdb_intensity_cap"] = panel_g2["gsdb_intensity"].clip(upper=6)

panel_g2["gsdb_intensity_cap"].max()
panel_g2.loc[panel_g2["gsdb_intensity_cap"]==6, ["COUNTERPART_COUNTRY.ID","year"]].head()

Panel UN

1) Trade balance (Exports − Imports)

In [ ]:
# panel_un = panel_un.copy()
panel_un["Exports of goods"] = pd.to_numeric(panel_un["Exports of goods"], errors="coerce").fillna(0)
panel_un["Imports of goods"] = pd.to_numeric(panel_un["Imports of goods"], errors="coerce").fillna(0)

panel_un["trade_balance"] = panel_un["Exports of goods"] - panel_un["Imports of goods"]

2) Annual % change for exports and imports (YoY %)

In [ ]:
#Rule: if the value in the previous year is 0 or missing → the result will be NaN (to avoid infinities).

panel_un["year"] = panel_un["year"].astype(int)

panel_un = panel_un.sort_values(["COUNTERPART_COUNTRY.ID", "year"])

panel_un["exp_lag"] = panel_un.groupby("COUNTERPART_COUNTRY.ID")["Exports of goods"].shift(1)
panel_un["imp_lag"] = panel_un.groupby("COUNTERPART_COUNTRY.ID")["Imports of goods"].shift(1)

panel_un["exp_yoy_pct"] = np.where(
    panel_un["exp_lag"] > 0,
    (panel_un["Exports of goods"] - panel_un["exp_lag"]) / panel_un["exp_lag"] * 100,
    np.nan
)

panel_un["imp_yoy_pct"] = np.where(
    panel_un["imp_lag"] > 0,
    (panel_un["Imports of goods"] - panel_un["imp_lag"]) / panel_un["imp_lag"] * 100,
    np.nan
)

3) Trade share each year (for exports and imports separately)

In [ ]:
#exp_total_year = sum of exports across all partners for that year
#imp_total_year = sum of imports across all partners for that year

year_totals = (panel_un.groupby("year")[["Exports of goods", "Imports of goods"]]
               .sum()
               .rename(columns={"Exports of goods":"exp_total_year",
                                "Imports of goods":"imp_total_year"}))

panel_un = panel_un.merge(year_totals, on="year", how="left")

panel_un["exp_share"] = np.where(
    panel_un["exp_total_year"] > 0,
    panel_un["Exports of goods"] / panel_un["exp_total_year"],
    np.nan
)

panel_un["imp_share"] = np.where(
    panel_un["imp_total_year"] > 0,
    panel_un["Imports of goods"] / panel_un["imp_total_year"],
    np.nan
)

In [ ]:
# column names
print(panel_un.columns.tolist())

# first 10 raws
panel_un.head(10)

In [ ]:
#check
panel_un.groupby("year")["exp_share"].sum().head()
panel_un.groupby("year")["imp_share"].sum().head()

In [ ]:
panel_un.to_csv(r"C:\Users\psimo\Music\Python\Paper\RU_trade_enriched_2016_2024.csv",
                index=False, encoding="utf-8-sig")

In [ ]:
# Mapping: ISO3 -> country name (take from panel_un for consistency)
country_map = (
    panel_un[["COUNTERPART_COUNTRY.ID", "COUNTERPART_COUNTRY"]]
    .dropna()
    .drop_duplicates(subset=["COUNTERPART_COUNTRY.ID"])
    .set_index("COUNTERPART_COUNTRY.ID")["COUNTERPART_COUNTRY"]
    .to_dict()
)

1) Export partners who account for an average of 75% (pre vs post)

In [ ]:
def top75_table(df, share_col, years, threshold=0.75):
    tmp = df[df["year"].isin(years)].copy()

    avg = (tmp.groupby("COUNTERPART_COUNTRY.ID", as_index=False)[share_col]
           .mean()
           .rename(columns={share_col: "avg_share"}))

    avg = avg.sort_values("avg_share", ascending=False)
    avg["cum_share"] = avg["avg_share"].cumsum()

    selected = avg[avg["cum_share"] <= threshold].copy()
# add 1 more raw to really exceed 75%
    if len(selected) < len(avg):
        selected = pd.concat([selected, avg.iloc[[len(selected)]]], ignore_index=True)

    return selected

pre_years  = list(range(2016, 2022))  # 2016–2021
post_years = list(range(2022, 2025))  # 2022–2024

exp_pre  = top75_table(panel_un, "exp_share", pre_years, threshold=0.75)
exp_post = top75_table(panel_un, "exp_share", post_years, threshold=0.75)

# merge into one table according to ISO3
exp = exp_pre.merge(exp_post, on="COUNTERPART_COUNTRY.ID", how="outer", suffixes=("_pre", "_post"))

# add name from mapping (not from join with period tables)
exp["Country"] = exp["COUNTERPART_COUNTRY.ID"].map(country_map)

# in percent and 2 decimal places
for c in ["avg_share_pre", "cum_share_pre", "avg_share_post", "cum_share_post"]:
    exp[c] = (exp[c] * 100).round(2)

# nice columns + neat layout
exp = exp.rename(columns={
    "COUNTERPART_COUNTRY.ID": "Partner (ISO3)",
    "avg_share_pre":  "Export avg share (2016–2021), %",
    "cum_share_pre":  "Export cum share (2016–2021), %",
    "avg_share_post": "Export avg share (2022–2024), %",
    "cum_share_post": "Export cum share (2022–2024), %",
})

# sort: first by pre avg, and if NaN then by post avg
exp["__sort"] = exp["Export avg share (2016–2021), %"].fillna(-1)
exp = exp.sort_values(["__sort", "Export avg share (2022–2024), %"], ascending=[False, False]).drop(columns="__sort")

# without index display
exp_table = exp[[
    "Partner (ISO3)", "Country",
    "Export avg share (2016–2021), %", "Export cum share (2016–2021), %",
    "Export avg share (2022–2024), %", "Export cum share (2022–2024), %",
]].reset_index(drop=True)

pct_cols = [c for c in exp_table.columns if "%" in c]
display(
    exp_table.style
      .hide(axis="index")
      .format({c: "{:.2f}" for c in pct_cols})
)

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper\export_partners_75_pre_post.csv"

exp_table.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved:", out_path)

2) Import partners who average 75% (pre vs post)

In [ ]:
imp_pre  = top75_table(panel_un, "imp_share", pre_years, threshold=0.75)
imp_post = top75_table(panel_un, "imp_share", post_years, threshold=0.75)

imp = imp_pre.merge(imp_post, on="COUNTERPART_COUNTRY.ID", how="outer", suffixes=("_pre", "_post"))
imp["Country"] = imp["COUNTERPART_COUNTRY.ID"].map(country_map)

for c in ["avg_share_pre", "cum_share_pre", "avg_share_post", "cum_share_post"]:
    imp[c] = (imp[c] * 100).round(2)

imp = imp.rename(columns={
    "COUNTERPART_COUNTRY.ID": "Partner (ISO3)",
    "avg_share_pre":  "Import avg share (2016–2021), %",
    "cum_share_pre":  "Import cum share (2016–2021), %",
    "avg_share_post": "Import avg share (2022–2024), %",
    "cum_share_post": "Import cum share (2022–2024), %",
})

imp["__sort"] = imp["Import avg share (2016–2021), %"].fillna(-1)
imp = imp.sort_values(["__sort", "Import avg share (2022–2024), %"], ascending=[False, False]).drop(columns="__sort")

imp_table = imp[[
    "Partner (ISO3)", "Country",
    "Import avg share (2016–2021), %", "Import cum share (2016–2021), %",
    "Import avg share (2022–2024), %", "Import cum share (2022–2024), %",
]].reset_index(drop=True)

pct_cols = [c for c in imp_table.columns if "%" in c]
display(
    imp_table.style
      .hide(axis="index")
      .format({c: "{:.2f}" for c in pct_cols})
)

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper\import_partners_75_pre_post.csv"

imp_table.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved:", out_path)

Top partners are defined as the smallest set of countries that jointly accounts for at least 75% of Russia’s total exports (imports), computed using average annual partner shares over 2016–2021 and over 2022–2024.

3) What came in/out (export pre vs post; import pre vs post)

In [ ]:
# --- EXPORT: entry/exit ---
exp_pre_set = set(exp_pre["COUNTERPART_COUNTRY.ID"])
exp_post_set = set(exp_post["COUNTERPART_COUNTRY.ID"])

exp_enter = sorted(exp_post_set - exp_pre_set)
exp_exit  = sorted(exp_pre_set - exp_post_set)

# nice names
id_to_name = dict(panel_un.drop_duplicates("COUNTERPART_COUNTRY.ID")
                  .set_index("COUNTERPART_COUNTRY.ID")["COUNTERPART_COUNTRY"])

exp_enter_df = pd.DataFrame({"COUNTERPART_COUNTRY.ID": exp_enter})
exp_enter_df["COUNTERPART_COUNTRY"] = exp_enter_df["COUNTERPART_COUNTRY.ID"].map(id_to_name)

exp_exit_df = pd.DataFrame({"COUNTERPART_COUNTRY.ID": exp_exit})
exp_exit_df["COUNTERPART_COUNTRY"] = exp_exit_df["COUNTERPART_COUNTRY.ID"].map(id_to_name)

print("Export: entered (post not in pre) =", len(exp_enter_df))
print("Export: exited  (pre not in post) =", len(exp_exit_df))

display(exp_enter_df)
display(exp_exit_df)

In [ ]:
# --- IMPORT: entry/exit ---
imp_pre_set = set(imp_pre["COUNTERPART_COUNTRY.ID"])
imp_post_set = set(imp_post["COUNTERPART_COUNTRY.ID"])

imp_enter = sorted(imp_post_set - imp_pre_set)
imp_exit  = sorted(imp_pre_set - imp_post_set)

imp_enter_df = pd.DataFrame({"COUNTERPART_COUNTRY.ID": imp_enter})
imp_enter_df["COUNTERPART_COUNTRY"] = imp_enter_df["COUNTERPART_COUNTRY.ID"].map(id_to_name)

imp_exit_df = pd.DataFrame({"COUNTERPART_COUNTRY.ID": imp_exit})
imp_exit_df["COUNTERPART_COUNTRY"] = imp_exit_df["COUNTERPART_COUNTRY.ID"].map(id_to_name)

print("Import: entered (post not in pre) =", len(imp_enter_df))
print("Import: exited  (pre not in post) =", len(imp_exit_df))

display(imp_enter_df)
display(imp_exit_df)

4) Joint list of strategic partners and sample size

In [ ]:
# union over all 4 sets
union_ids = sorted(exp_pre_set | exp_post_set | imp_pre_set | imp_post_set)

partners_union = pd.DataFrame({"COUNTERPART_COUNTRY.ID": union_ids})
partners_union["COUNTERPART_COUNTRY"] = partners_union["COUNTERPART_COUNTRY.ID"].map(id_to_name)


# 1) nice names
partners_union_pretty = partners_union.rename(columns={
    "COUNTERPART_COUNTRY.ID": "Partner (ISO3)",
    "COUNTERPART_COUNTRY": "Partner country"
}).copy()

# 2) display without index in Jupyter environment
display(
    partners_union_pretty.style
    .hide(axis="index")
)


filename = "strategic_partners_union_25.csv"
save_path = os.path.join(out_folder, filename)

partners_union_pretty.to_csv(save_path, index=False)
print("Saved:", save_path)

# change folder
out_folder = r"C:\Users\psimo\Music\Python\Paper"
out_path = out_folder + r"\RU_strategic_partners_union_75pct.csv"

partners_union.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved to:", out_path)

In [ ]:
# 1) leave only strategic partners (union 75%)
strategic_ids = set(partners_union["COUNTERPART_COUNTRY.ID"])

panel_strategic = panel_un[panel_un["COUNTERPART_COUNTRY.ID"].isin(strategic_ids)].copy()

print("Rows:", panel_un.shape[0], "->", panel_strategic.shape[0])
print("Unique partners:", panel_strategic["COUNTERPART_COUNTRY.ID"].nunique())

# 2) sort nicely
panel_strategic = panel_strategic.sort_values(["COUNTERPART_COUNTRY.ID","year"])


out_path = r"C:\Users\psimo\Music\Python\Paper\RU_trade_strategic_sample_75pct_2016_2024.csv"
panel_strategic.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved to:", out_path)

1) Check that panel_strategic has what it needs

In [ ]:
req_cols = ["COUNTERPART_COUNTRY.ID","year","exp_share","imp_share"]
missing = [c for c in req_cols if c not in panel_strategic.columns]
print("Missing in panel_strategic:", missing)
panel_strategic[req_cols].head()

In [ ]:
print(gsdb_it2.columns.tolist())
gsdb_it2.head()

3) Merge: panel_strategic + gsdb_it2 (2016–2023)

In [ ]:
# 2016–2023, because GSDB until 2023
panel_16_23 = panel_strategic[panel_strategic["year"].between(2016, 2023)].copy()
gsdb_16_23  = gsdb_it2[gsdb_it2["year"].between(2016, 2023)].copy()

# take only the necessary columns and do the merge
df = panel_16_23.merge(
    gsdb_16_23[["COUNTERPART_COUNTRY.ID", "year", "gsdb_intensity"]],
    on=["COUNTERPART_COUNTRY.ID", "year"],
    how="left"
)

# if there is no sanction record -> 0
df["gsdb_intensity"] = df["gsdb_intensity"].fillna(0).astype(int)

print("Rows:", df.shape[0], "Partners:", df["COUNTERPART_COUNTRY.ID"].nunique())
df[["COUNTERPART_COUNTRY.ID","year","exp_share","imp_share","gsdb_intensity"]].head(10)

4) Check: Does gsdb_intensity have variation?

In [ ]:
# are there any non-zeros and how are they distributed
print("Share of non-zero gsdb_intensity:", (df["gsdb_intensity"] > 0).mean())
print("Unique values:", sorted(df["gsdb_intensity"].unique().tolist()))

# top countries by average intensity
(df.groupby("COUNTERPART_COUNTRY.ID")["gsdb_intensity"]
   .mean()
   .sort_values(ascending=False)
   .head(15))

In [ ]:
# are USA/Poland/Latvia even in gsdb_it2 as a sanctioning_state?
targets = ["United States", "Poland", "Latvia", "Lithuania", "Estonia"]
gsdb_it2[gsdb_it2["sanctioning_state"].isin(targets)][["sanctioning_state","year","gsdb_intensity","COUNTERPART_COUNTRY.ID"]].head(30)

In [ ]:
# how many unique authorized ones does ISO 3 have and how many do not haveprint("Unique sanctioning_state:", gsdb_it2["sanctioning_state"].nunique())
print("Unique ISO3 mapped:", gsdb_it2["COUNTERPART_COUNTRY.ID"].nunique())
print("Missing ISO3 rows:", gsdb_it2["COUNTERPART_COUNTRY.ID"].isna().mean())

A) Minimum coalition split only in EU top25 + basic split

In [ ]:
EU_TOP25 = ["BEL","CZE","DEU","FIN","FRA","ITA","LVA","NLD","POL"]

def norm_token(s: str) -> str:
    return str(s).strip()

def expand_sanctioners(row):
    """
    Iš vienos GSDB eilutės grąžina sąrašą ISO3 sankcionuotojų (partnerių),
    kuriems priskiriame tos eilutės gsdb_intensity.
    Minimalus A variantas:
      - jei yra "EU" -> grąžina EU_TOP25
      - kitaip split per kablelius ir konvertuoja kiekvieną tokeną į ISO3 (jei įmanoma)
    """
    s = str(row["sanctioning_state"]).replace("Korea, South", "South Korea")
    tokens = [norm_token(t) for t in s.split(",")]  # split coalitions into parts
    tokens = [t for t in tokens if t]               # throws empty
    
    iso3_list = []

   # 1) if there is EU (in any position) -> assign to EU top25
    if any(t.upper() == "EU" for t in tokens):
        iso3_list.extend(EU_TOP25)

    # 2) other tokens: converted to ISO3
# (here the "EU" token is intentionally ignored, as it has already been converted to EU countries)
    other = [t for t in tokens if t.upper() != "EU"]
    other = [t for t in other if t.upper() not in {"G7"}]

    # manual fixes for several common cases
    manual = {
        "United States": "USA",
        "United Kingdom": "GBR",
        "Korea, South": "KOR",
        "South Korea": "KOR",
        "Czech Republic": "CZE",
        "Russia": "RUS",
    }

    for t in other:
        if t in manual:
            iso3_list.append(manual[t])
        else:
            code = coco.convert(names=t, to="ISO3", not_found=None)
            if code is not None and isinstance(code, str) and code != "not found":
                iso3_list.append(code)

    # uniquely (without duplicates)
    return sorted(set(iso3_list))

# 1) split rows into (ISO3, year, gsdb_intensity)
rows = []
for _, r in gsdb_it2.iterrows():
    iso3s = expand_sanctioners(r)
    for c in iso3s:
        rows.append((c, int(r["year"]), int(r["gsdb_intensity"])))

expanded = pd.DataFrame(rows, columns=["COUNTERPART_COUNTRY.ID","year","gsdb_intensity"])

# 2) aggregate to partner-year: MAX and cap=6
gsdb_ready = (expanded.groupby(["COUNTERPART_COUNTRY.ID","year"], as_index=False)["gsdb_intensity"]
                     .max())
gsdb_ready["gsdb_intensity_cap"] = gsdb_ready["gsdb_intensity"].clip(upper=6).astype(int)

# 3) quick checks for the top 25 most important
check_ids = ["USA","GBR","JPN","KOR","POL","LVA","DEU","FRA","ITA","NLD","BEL","CZE","FIN"]
print(gsdb_ready[gsdb_ready["COUNTERPART_COUNTRY.ID"].isin(check_ids)]
      .groupby("COUNTERPART_COUNTRY.ID")["gsdb_intensity_cap"].mean()
      .sort_values(ascending=False))

Sanity check: 1) Show the USA and one EU country per year (or increase in 2022–2023)

In [ ]:
gsdb_ready.loc[gsdb_ready["COUNTERPART_COUNTRY.ID"].isin(["USA","DEU"]), 
               ["COUNTERPART_COUNTRY.ID","year","gsdb_intensity_cap"]].sort_values(["COUNTERPART_COUNTRY.ID","year"])

Sanity check 2) Check if the EU top25 countries have the same values in the same year

In [ ]:
EU_TOP25 = ["BEL","CZE","DEU","FIN","FRA","ITA","LVA","NLD","POL"]

(gsdb_ready[gsdb_ready["COUNTERPART_COUNTRY.ID"].isin(EU_TOP25)]
 .groupby("year")["gsdb_intensity_cap"]
 .nunique()
 .sort_index())

4) Minimal check whether gsdb_it2 is really a “Russia-only target”

In [ ]:
gsdb_ru["sanctioned_state"].value_counts().head(30)

To avoid conflating multi-target sanction regimes, we restrict the sanctions sample to episodes where Russia is the sole sanctioned state.

Filter only “Russia” and rebuild the entire pipeline

In [ ]:
gsdb_ru_only = gsdb_ru[gsdb_ru["sanctioned_state"].astype(str).str.strip() == "Russia"].copy()
gsdb_ru_multi = gsdb_ru[gsdb_ru["sanctioned_state"].astype(str).str.strip() != "Russia"].copy()

print("Russia-only rows:", gsdb_ru_only.shape[0])
print("Multi-target rows:", gsdb_ru_multi.shape[0])
gsdb_ru_multi["sanctioned_state"].value_counts().head(10)

In [ ]:
print(gsdb_it2.shape)

In [ ]:
print(gsdb_ru.columns.tolist())
gsdb_ru.head()

GSDB RECONSTRUCTION TO RU_ONLY

1) Create a year panel (2016–2023) from case-level with gsdb_intensity

In [ ]:
# 1) Russia-only
gsdb_ru_only = gsdb_ru[gsdb_ru["sanctioned_state"].astype(str).str.strip() == "Russia"].copy()

# 2) create copies of the raws for each year the episode is active
rows = []
for _, r in gsdb_ru_only.iterrows():
    b = int(r["begin"])
    e = int(r["end"]) if pd.notna(r["end"]) else 2023  # if end NaN, keep until 2023
    # restrict to study year
    for y in range(max(2016, b), min(2023, e) + 1):
        # intensity = how many categories are active (0–6)
        intensity = int(r["trade"]) + int(r["financial"]) + int(r["arms"]) + int(r["military"]) + int(r["travel"]) + int(r["other"])
        rows.append((r["sanctioning_state"], y, intensity))

gsdb_it2_ruonly = pd.DataFrame(rows, columns=["sanctioning_state","year","gsdb_intensity"])

print(gsdb_it2_ruonly.shape)
gsdb_it2_ruonly.head()

2) EU -> 9 EU top25, coalition split

In [ ]:
EU_TOP25 = ["BEL","CZE","DEU","FIN","FRA","ITA","LVA","NLD","POL"]

def norm_token(s): 
    return str(s).strip()

manual = {
    "United States": "USA",
    "United Kingdom": "GBR",
    "South Korea": "KOR",
    "Korea, South": "KOR",
    "Czech Republic": "CZE",
}

def expand_sanctioners(s):
    s = str(s).replace("Korea, South", "South Korea")
    tokens = [norm_token(t) for t in s.split(",") if norm_token(t)]
    iso3_list = []

    # EU token -> ES top25
    if any(t.upper() == "EU" for t in tokens):
        iso3_list.extend(EU_TOP25)

    # convert other tokens
    for t in tokens:
        if t.upper() in {"EU", "G7"}:
            continue
        if t in manual:
            iso3_list.append(manual[t])
        else:
            code = coco.convert(names=t, to="ISO3", not_found=None)
            if code is not None and isinstance(code, str) and code != "not found":
                iso3_list.append(code)

    return sorted(set(iso3_list))

rows = []
for _, r in gsdb_it2_ruonly.iterrows():
    iso3s = expand_sanctioners(r["sanctioning_state"])
    for c in iso3s:
        rows.append((c, int(r["year"]), int(r["gsdb_intensity"])))

expanded = pd.DataFrame(rows, columns=["COUNTERPART_COUNTRY.ID","year","gsdb_intensity"])

# aggregate: if multiple entries for the same country in the same year -> MAX
gsdb_ready = (expanded.groupby(["COUNTERPART_COUNTRY.ID","year"], as_index=False)["gsdb_intensity"]
                     .max())

# cap (although it should be 0–6 here, but for security reasons)
gsdb_ready["gsdb_intensity_cap"] = gsdb_ready["gsdb_intensity"].clip(upper=6).astype(int)

# sanity
check_ids = ["USA","GBR","JPN","KOR","POL","LVA","DEU","FRA","ITA","NLD","BEL","CZE","FIN"]
print(gsdb_ready[gsdb_ready["COUNTERPART_COUNTRY.ID"].isin(check_ids)]
      .groupby("COUNTERPART_COUNTRY.ID")["gsdb_intensity_cap"].mean()
      .sort_values(ascending=False))

3) Merge to panel (2016–2023)

In [ ]:
# 1) 2016–2023 (GSDB to 2023)
panel_16_23 = panel_strategic[panel_strategic["year"].between(2016, 2023)].copy()

# 2) Merge with sanctions intensity (cap)
df = panel_16_23.merge(
    gsdb_ready[["COUNTERPART_COUNTRY.ID", "year", "gsdb_intensity_cap"]],
    on=["COUNTERPART_COUNTRY.ID", "year"],
    how="left"
)

# 3) If there is no sanction record -> 0
df["gsdb_intensity_cap"] = df["gsdb_intensity_cap"].fillna(0).astype(int)

print("Rows:", df.shape[0], "Partners:", df["COUNTERPART_COUNTRY.ID"].nunique())
print("Non-zero share (intensity>0):", (df["gsdb_intensity_cap"] > 0).mean().round(3))

# 4) Appendix-ready base table
appendix_tbl = df[[
    "COUNTERPART_COUNTRY.ID", "year", "exp_share", "imp_share", "gsdb_intensity_cap"
]].copy()

appendix_tbl = appendix_tbl.rename(columns={
    "COUNTERPART_COUNTRY.ID": "Partner (ISO3)",
    "year": "Year",
    "exp_share": "Export share (0–1)",
    "imp_share": "Import share (0–1)",
    "gsdb_intensity_cap": "Sanctions intensity (cap, 0–6)"
})

appendix_tbl = (appendix_tbl
    .sort_values(["Partner (ISO3)", "Year"])
    .reset_index(drop=True)
)


display(appendix_tbl)


appendix_tbl.to_csv(f"{out_folder}/appendix_partner_year_shares_intensity_2016_2023.csv", index=False)

In [ ]:
# Mini table: 1 row/partner (2016-2023)
mini = df.copy()

# 1) first year when intensity>0 (if never -> NaN)
first_year = (mini.loc[mini["gsdb_intensity_cap"] > 0]
              .groupby("COUNTERPART_COUNTRY.ID")["year"]
              .min())

# 2) summary by country
mini_tbl = (mini.groupby("COUNTERPART_COUNTRY.ID", as_index=False)
            .agg(
                avg_export_share=("exp_share", "mean"),
                avg_import_share=("imp_share", "mean"),
                avg_intensity=("gsdb_intensity_cap", "mean"),
                max_intensity=("gsdb_intensity_cap", "max"),
                nonzero_years=("gsdb_intensity_cap", lambda x: int((x > 0).sum())),
                n_years=("year", "nunique"),
            ))

mini_tbl["first_sanction_year"] = mini_tbl["COUNTERPART_COUNTRY.ID"].map(first_year)

# 3) formating
mini_tbl = mini_tbl.rename(columns={
    "COUNTERPART_COUNTRY.ID": "Partner (ISO3)",
    "first_sanction_year": "First sanction year",
    "avg_intensity": "Avg sanctions intensity (0–6)",
    "max_intensity": "Max sanctions intensity (0–6)",
    "avg_export_share": "Avg export share (0–1)",
    "avg_import_share": "Avg import share (0–1)",
    "nonzero_years": "Years with intensity>0",
    "n_years": "Years observed (2016–2023)"
})

# roundings: shares to 4 decimal places, intensity to 2
mini_tbl["Avg export share (0–1)"] = mini_tbl["Avg export share (0–1)"].round(4)
mini_tbl["Avg import share (0–1)"] = mini_tbl["Avg import share (0–1)"].round(4)
mini_tbl["Avg sanctions intensity (0–6)"] = mini_tbl["Avg sanctions intensity (0–6)"].round(2)

# nice sorting
mini_tbl = (mini_tbl
            .sort_values(["Max sanctions intensity (0–6)", "Avg sanctions intensity (0–6)"],
                         ascending=False)
            .reset_index(drop=True))

display(mini_tbl)

mini_tbl.to_csv(f"{out_folder}/appendix_mini_partner_summary_2016_2023.csv", index=False)

1) Export model (exp_share)

In [ ]:
# Rename column
df = df.rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"})

In [ ]:
m_exp = smf.ols(
    "exp_share ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["partner_id"]})

print(m_exp.summary().tables[1])
print("R-squared:", round(m_exp.rsquared, 4))
print("Adj. R-squared:", round(m_exp.rsquared_adj, 4))
print("N:", int(m_exp.nobs))

In [ ]:
out_path = Path(r"C:\Users\psimo\Music\Python\Paper")
out_path.mkdir(parents=True, exist_ok=True)


fig_path = out_path / "fig_share_trends_by_sanctioning.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print("Saved:", fig_path)

2) Import model (imp_share)

In [ ]:
m_imp = smf.ols(
    "imp_share ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["partner_id"]})

print(m_imp.summary().tables[1])
print("R-squared:", round(m_imp.rsquared, 4))
print("Adj. R-squared:", round(m_imp.rsquared_adj, 4))
print("N:", int(m_imp.nobs))

4) Robustness

Exclude Ukraine (UKR)

Ukraine is a "special case" - the trade structure and sanctions regime are extreme, and can create leverage.

In [ ]:
df_no_ukr = df[df["partner_id"] != "UKR"].copy()

m_exp_no_ukr = smf.ols(
    "exp_share ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df_no_ukr
).fit(cov_type="cluster", cov_kwds={"groups": df_no_ukr["partner_id"]})

print(m_exp_no_ukr.summary().tables[1])
print("R-squared:", round(m_exp_no_ukr.rsquared, 4))
print("Adj. R-squared:", round(m_exp_no_ukr.rsquared_adj, 4))
print("N:", int(m_exp_no_ukr.nobs))

Import share robustness: exclude UKR

In [ ]:
df_no_ukr = df[df["partner_id"] != "UKR"].copy()

m_imp_no_ukr = smf.ols(
    "imp_share ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df_no_ukr
).fit(cov_type="cluster", cov_kwds={"groups": df_no_ukr["partner_id"]})

print(m_imp_no_ukr.summary().tables[1])
print("R-squared:", round(m_imp_no_ukr.rsquared, 4))
print("Adj. R-squared:", round(m_imp_no_ukr.rsquared_adj, 4))
print("N:", int(m_imp_no_ukr.nobs))

1) Graph: average exp_share and imp_share by “sanctioning” (intensity>0) per year

In [ ]:
# 2×2 plot in percentages (%):
# Export/Import × (2016–2023 main sample with sanctions info) vs (2016–2024 trade-only descriptives)
# Requires: panel_strategic (2016–2024) and df (2016–2023 with gsdb_intensity_cap) + matplotlib

out_path = r"C:\Users\psimo\Music\Python\Paper"
out_file = "fig_2x2_avg_shares_sanctioning_2016_2024.png"
out_full = os.path.join(out_path, out_file)

# make sure that df has partner_id
if "partner_id" not in df.columns:
    df = df.rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"})

# 1) partner group: was the intensity of sanctions >0 at least once in 2016–2023?
sanction_flag = (df.groupby("partner_id")["gsdb_intensity_cap"].max() > 0).astype(int)

# 2) assign a group in the trade panel
tmp = panel_strategic.copy()
tmp["sanctioning"] = tmp["COUNTERPART_COUNTRY.ID"].map(sanction_flag).fillna(0).astype(int)

def make_avg_table(data, years):
    d = data[data["year"].between(years[0], years[1])].copy()
    avg = (d.groupby(["year", "sanctioning"], as_index=False)[["exp_share","imp_share"]].mean())
    # įto percentage
    avg["exp_share"] *= 100
    avg["imp_share"] *= 100
    return avg

avg_16_23 = make_avg_table(tmp, (2016, 2023))
avg_16_24 = make_avg_table(tmp, (2016, 2024))

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=False, sharey=False)

# ---------- EXPORT: 2016–2023 ----------
pivot = avg_16_23.pivot(index="year", columns="sanctioning", values="exp_share")
pivot.plot(marker="o", ax=axes[0,0])
axes[0,0].set_title("Export (main sample): 2016–2023 (linked to sanctions data)")
axes[0,0].set_xlabel("Year")
axes[0,0].set_ylabel("Average export share (%)")
axes[0,0].legend(["Non-sanctioning (0)", "Sanctioning (>0)"])

# --- EXPORT: 2016–2024  ---
pivot = avg_16_24.pivot(index="year", columns="sanctioning", values="exp_share")
axes[0,1].set_title("Export (trade-only descriptives): 2016–2024")
axes[0,1].set_xlabel("Year")
axes[0,1].set_ylabel("Average export share (%)")

color_map = {0: "C0", 1: "C1"}
label_map = {0: "Non-sanctioning (0)", 1: "Sanctioning (>0)"}

for col in [0, 1]:
    s = pivot[col].dropna()

    # 2016–2023 
    s_pre = s.loc[s.index <= 2023]
    axes[0,1].plot(
        s_pre.index, s_pre.values,
        marker="o",
        color=color_map[col],
        label=label_map[col]
    )

    # 2023→2024 
    if 2023 in s.index and 2024 in s.index:
        axes[0,1].plot(
            [2023, 2024], [s.loc[2023], s.loc[2024]],
            marker="o",
            linestyle="--",
            color=color_map[col]
        )

axes[0,1].legend()

# ---------- IMPORT: 2016–2023 ----------
pivot = avg_16_23.pivot(index="year", columns="sanctioning", values="imp_share")
pivot.plot(marker="o", ax=axes[1,0])
axes[1,0].set_title("Import (main sample): 2016–2023 (linked to sanctions data)")
axes[1,0].set_xlabel("Year")
axes[1,0].set_ylabel("Average import share (%)")
axes[1,0].legend(["Non-sanctioning (0)", "Sanctioning (>0)"])

# --- IMPORT: 2016–2024 ---
pivot = avg_16_24.pivot(index="year", columns="sanctioning", values="imp_share")
axes[1,1].set_title("Import (trade-only descriptives): 2016–2024")
axes[1,1].set_xlabel("Year")
axes[1,1].set_ylabel("Average import share (%)")

for col in [0, 1]:
    s = pivot[col].dropna()

    s_pre = s.loc[s.index <= 2023]
    axes[1,1].plot(
        s_pre.index, s_pre.values,
        marker="o",
        color=color_map[col],
        label=label_map[col]
    )

    if 2023 in s.index and 2024 in s.index:
        axes[1,1].plot(
            [2023, 2024], [s.loc[2023], s.loc[2024]],
            marker="o",
            linestyle="--",
            color=color_map[col]
        )

axes[1,1].legend()


fig.tight_layout()
fig.savefig(out_full, dpi=300, bbox_inches="tight")
print("Saved:", out_full)

plt.show()

In [ ]:
# 1) partner group: was the intensity of sanctions >0 at least once in 2016–2023?
sanction_flag = (df.groupby("partner_id")["gsdb_intensity_cap"].max() > 0).astype(int)

# 2) trade panel with group
tmp = panel_strategic.copy()
tmp["sanctioning"] = tmp["COUNTERPART_COUNTRY.ID"].map(sanction_flag).fillna(0).astype(int)

# 3) long table (year × group) with means and N
long = (tmp.groupby(["year", "sanctioning"], as_index=False)
          .agg(
              exp_share_pct=("exp_share", lambda x: x.mean() * 100),
              imp_share_pct=("imp_share", lambda x: x.mean() * 100),
              N=("COUNTERPART_COUNTRY.ID", "nunique")
          ))

# 4) wide (years in rows, 0/1 in columns)
wide = long.pivot(index="year", columns="sanctioning", values=["exp_share_pct", "imp_share_pct", "N"])

wide.columns = [
    f"{metric}_{'non_sanctioning' if grp == 0 else 'sanctioning'}"
    for metric, grp in wide.columns
]
wide = wide.reset_index().sort_values("year")

# 5) nice names
wide_pretty = wide.rename(columns={
    "year": "Year",
    "exp_share_pct_non_sanctioning": "Export share (%) — Non-sanctioning",
    "exp_share_pct_sanctioning":     "Export share (%) — Sanctioning",
    "imp_share_pct_non_sanctioning": "Import share (%) — Non-sanctioning",
    "imp_share_pct_sanctioning":     "Import share (%) — Sanctioning",
    "N_non_sanctioning":             "N partners — Non-sanctioning",
    "N_sanctioning":                 "N partners — Sanctioning",
})

# 6) formating
pct_cols = [
    "Export share (%) — Non-sanctioning",
    "Export share (%) — Sanctioning",
    "Import share (%) — Non-sanctioning",
    "Import share (%) — Sanctioning",
]
n_cols = ["N partners — Non-sanctioning", "N partners — Sanctioning"]

display(
    wide_pretty.style
      .hide(axis="index")
      .format({c: "{:.2f}" for c in pct_cols} | {c: "{:.0f}" for c in n_cols})
)


out_path = r"C:\Users\psimo\Music\Python\Paper"
os.makedirs(out_path, exist_ok=True)

# 1) Excel 
excel_file = os.path.join(out_path, "Table_group_means_sanctioning_2016_2024.xlsx")
wide_pretty.to_excel(excel_file, index=False)

# 2) CSV 
csv_file = os.path.join(out_path, "Table_group_means_sanctioning_2016_2024.csv")
wide_pretty.to_csv(csv_file, index=False, encoding="utf-8-sig")

print("Saved:")
print(excel_file)
print(csv_file)

2)TOP15 dumbbell 2×2 (with CHN vs excluding CHN)

In [ ]:
def build_top_prepost(panel, flow_col, top_n=15):
    pre = panel[panel["year"].between(2016, 2021)].groupby("COUNTERPART_COUNTRY.ID")[flow_col].mean()
    post = panel[panel["year"].between(2022, 2023)].groupby("COUNTERPART_COUNTRY.ID")[flow_col].mean()

    comp = pd.concat([pre, post], axis=1)
    comp.columns = ["pre", "post"]
    comp = comp.fillna(0)
    comp["avg_prepost"] = (comp["pre"] + comp["post"]) / 2

    top = comp.sort_values("avg_prepost", ascending=False).head(top_n).copy()
    top["pre_pct"] = top["pre"] * 100
    top["post_pct"] = top["post"] * 100
    top = top.sort_values("post_pct")
    return top

def dumbbell_clean(ax, top, xlabel, title, xlim=None):
    y = np.arange(len(top))

    ax.hlines(y=y, xmin=top["pre_pct"], xmax=top["post_pct"], linewidth=1)

    # PRE: hollow
    ax.scatter(top["pre_pct"], y, s=70, facecolors="none", edgecolors="black",
               linewidths=1.2, marker="o", label="2016–2021")
    # POST: filled
    ax.scatter(top["post_pct"], y, s=70, facecolors="black", edgecolors="black",
               linewidths=1.0, marker="o", label="2022–2023")

    ax.set_yticks(y)
    ax.set_yticklabels(top.index)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.legend(loc="lower right")

# --- TOP15 (with CHN) ---
top_exp = build_top_prepost(panel_strategic, "exp_share", top_n=15)
top_imp = build_top_prepost(panel_strategic, "imp_share", top_n=15)

# --- TOP15 (excluding CHN) ---
top_exp_nochn = top_exp.drop(index="CHN", errors="ignore").copy()
top_imp_nochn = top_imp.drop(index="CHN", errors="ignore").copy()

# rearrange after drop (so that there are no “holes”)
top_exp_nochn = top_exp_nochn.sort_values("post_pct")
top_imp_nochn = top_imp_nochn.sort_values("post_pct")

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

# EXPORT row
dumbbell_clean(
    axes[0,0], top_exp,
    xlabel="Export share (%)",
    title="Export TOP15: pre vs post (with China)"
)
dumbbell_clean(
    axes[0,1], top_exp_nochn,
    xlabel="Export share (%)",
    title="Export TOP15: pre vs post (excluding China, zoom)",
    xlim=(0, 12)  
)

# IMPORT row
dumbbell_clean(
    axes[1,0], top_imp,
    xlabel="Import share (%)",
    title="Import TOP15: pre vs post (with China)"
)
dumbbell_clean(
    axes[1,1], top_imp_nochn,
    xlabel="Import share (%)",
    title="Import TOP15: pre vs post (excluding China, zoom)",
    xlim=(0, 12)   
)

out_path = r"C:\Users\psimo\Music\Python\Paper"
os.makedirs(out_path, exist_ok=True)

out_file = "Fig_TOP15_dumbbell_2x2.png"
out_full = os.path.join(out_path, out_file)


fig.savefig(out_full, dpi=300, bbox_inches="tight")
print("Saved:", out_full)

plt.show()

In [ ]:
def make_top_table(panel, flow_col, top_n=25, pre_years=(2016, 2021), post_years=(2022, 2023)):
    pre = (panel[panel["year"].between(*pre_years)]
           .groupby("COUNTERPART_COUNTRY.ID")[flow_col].mean())
    post = (panel[panel["year"].between(*post_years)]
            .groupby("COUNTERPART_COUNTRY.ID")[flow_col].mean())

    comp = pd.concat([pre, post], axis=1)
    comp.columns = ["pre", "post"]
    comp = comp.fillna(0)

    comp["Pre (2016–2021), %"] = comp["pre"] * 100
    comp["Post (2022–2023), %"] = comp["post"] * 100
    comp["Δ (pp)"] = comp["Post (2022–2023), %"] - comp["Pre (2016–2021), %"]
    comp["Avg(pre,post), %"] = (comp["Pre (2016–2021), %"] + comp["Post (2022–2023), %"]) / 2

    top = (comp.sort_values("Avg(pre,post), %", ascending=False)
           .head(top_n)
           .reset_index()
           .rename(columns={"COUNTERPART_COUNTRY.ID": "Partner (ISO3)"}))

    top = top[["Partner (ISO3)", "Pre (2016–2021), %", "Post (2022–2023), %", "Δ (pp)", "Avg(pre,post), %"]].round(2)
    return top

export_top25 = make_top_table(panel_strategic, "exp_share", top_n=25)

display(
    export_top25.style
    .hide(axis="index")
    .format({
        "Pre (2016–2021), %": "{:.2f}",
        "Post (2022–2023), %": "{:.2f}",
        "Δ (pp)": "{:.2f}",
        "Avg(pre,post), %": "{:.2f}",
    })
)


export_top25.to_csv("appendix_export_top25_prepost.csv", index=False)

In [ ]:
def make_top_table(panel, flow_col, top_n=25, pre_years=(2016, 2021), post_years=(2022, 2023)):
    pre = (panel[panel["year"].between(*pre_years)]
           .groupby("COUNTERPART_COUNTRY.ID")[flow_col].mean())
    post = (panel[panel["year"].between(*post_years)]
            .groupby("COUNTERPART_COUNTRY.ID")[flow_col].mean())

    comp = pd.concat([pre, post], axis=1)
    comp.columns = ["pre", "post"]
    comp = comp.fillna(0)

    comp["Pre (2016–2021), %"] = comp["pre"] * 100
    comp["Post (2022–2023), %"] = comp["post"] * 100
    comp["Δ (pp)"] = comp["Post (2022–2023), %"] - comp["Pre (2016–2021), %"]
    comp["Avg(pre,post), %"] = (comp["Pre (2016–2021), %"] + comp["Post (2022–2023), %"]) / 2

    top = (comp.sort_values("Avg(pre,post), %", ascending=False)
           .head(top_n)
           .reset_index()
           .rename(columns={"COUNTERPART_COUNTRY.ID": "Partner (ISO3)"}))

    top = top[["Partner (ISO3)", "Pre (2016–2021), %", "Post (2022–2023), %", "Δ (pp)", "Avg(pre,post), %"]].round(2)
    return top

import_top25 = make_top_table(panel_strategic, "imp_share", top_n=25)

display(
    import_top25.style
    .hide(axis="index")
    .format({
        "Pre (2016–2021), %": "{:.2f}",
        "Post (2022–2023), %": "{:.2f}",
        "Δ (pp)": "{:.2f}",
        "Avg(pre,post), %": "{:.2f}",
    })
)


import_top25.to_csv("appendix_import_top25_prepost.csv", index=False)

TOP 15 pre vs post shares + intensity (post) + Δ, sorted by total_change_pp

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper"
out_file = "appendix_group_timing_3panels.png"
out_full = os.path.join(out_path, out_file)

# 1) first sanction year per partner (based on df which has partner_id, year, gsdb_intensity_cap)
tmp_s = df[df["year"].between(2016, 2023)].copy()
tmp_s = tmp_s[tmp_s["gsdb_intensity_cap"] > 0]

first_year = tmp_s.groupby("partner_id")["year"].min()  # series: partner_id -> first sanction year

# 2) timing group
partners = pd.Index(panel_strategic["COUNTERPART_COUNTRY.ID"].unique(), name="partner_id")
grp = pd.DataFrame({"partner_id": partners})
grp["first_sanction_year"] = grp["partner_id"].map(first_year)  # NaN = never

def classify(y):
    if pd.isna(y):
        return "Never sanctioning"
    elif y <= 2021:
        return "Early sanctioners (<=2021)"
    else:
        return "Late sanctioners (>=2022)"

grp["group_timing"] = grp["first_sanction_year"].apply(classify)

# 3) merge group into trade panel (2016–2024 trade-only)
trade = panel_strategic.copy()
trade = trade.rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"})
trade = trade.merge(grp[["partner_id", "group_timing"]], on="partner_id", how="left")

# 4) yearly averages by group (shares in %, intensity on 0–6 scale)
trade["exp_pct"] = trade["exp_share"] * 100
trade["imp_pct"] = trade["imp_share"] * 100

avg_trade = (trade.groupby(["year","group_timing"], as_index=False)[["exp_pct","imp_pct"]].mean())

avg_int = (df[df["year"].between(2016, 2023)]
             .groupby(["year","partner_id"], as_index=False)["gsdb_intensity_cap"].mean()
             .merge(grp[["partner_id","group_timing"]], on="partner_id", how="left")
             .groupby(["year","group_timing"], as_index=False)["gsdb_intensity_cap"].mean()
             .rename(columns={"gsdb_intensity_cap":"avg_intensity"}))

avg = avg_trade.merge(avg_int, on=["year","group_timing"], how="left")  # intensity will be NaN 2024

# 5) plot: 3 panels (Never / Early / Late)
order = ["Never sanctioning", "Early sanctioners (<=2021)", "Late sanctioners (>=2022)"]

fig, axes = plt.subplots(3, 1, figsize=(11, 12), constrained_layout=True, sharex=True)

for ax, g in zip(axes, order):
    d = avg[avg["group_timing"] == g].sort_values("year")

    # left axis: shares
    ax.plot(d["year"], d["exp_pct"], marker="o", label="Export share (%)")
    ax.plot(d["year"], d["imp_pct"], marker="o", label="Import share (%)")
    ax.set_ylabel("Share (%)")
    ax.set_title(g)

    # right axis: intensity (stops at 2023 naturally)
    ax2 = ax.twinx()
    ax2.plot(d["year"], d["avg_intensity"], linestyle="--", marker="o", label="Avg intensity (0–6)")
    ax2.set_ylabel("Avg sanctions intensity")

    # legends: combine
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left")

axes[-1].set_xlabel("Year")


os.makedirs(out_path, exist_ok=True)
fig.savefig(out_full, dpi=300, bbox_inches="tight")
print("Saved:", out_full)

plt.show()
plt.close(fig)

Appendix. Small multiples by groups (country-level, grouped)

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper"
os.makedirs(out_path, exist_ok=True)

# --- 1) groups (Never/Early/Late) ---
tmp_s = df[df["year"].between(2016, 2023)].copy()
tmp_s = tmp_s[tmp_s["gsdb_intensity_cap"] > 0]
first_year = tmp_s.groupby("partner_id")["year"].min()

partners = pd.Index(panel_strategic["COUNTERPART_COUNTRY.ID"].unique(), name="partner_id")
grp = pd.DataFrame({"partner_id": partners})
grp["first_sanction_year"] = grp["partner_id"].map(first_year)

def classify(y):
    if pd.isna(y):
        return "Never sanctioning"
    elif y <= 2021:
        return "Early sanctioners (<=2021)"
    else:
        return "Late sanctioners (>=2022)"

grp["group_timing"] = grp["first_sanction_year"].apply(classify)

# --- 2) country-year panel with exp/imp/intensity ---
trade = panel_strategic.copy().rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"})
trade["exp_pct"] = trade["exp_share"] * 100
trade["imp_pct"] = trade["imp_share"] * 100

int_by_country_year = (
    df[df["year"].between(2016, 2023)]
      .groupby(["partner_id", "year"], as_index=False)["gsdb_intensity_cap"]
      .mean()
      .rename(columns={"gsdb_intensity_cap": "intensity"})
)

panel_cy = (
    trade.merge(int_by_country_year, on=["partner_id", "year"], how="left")
         .merge(grp[["partner_id", "group_timing"]], on="partner_id", how="left")
)

# --- 3) function: mini graphs for each country in the group + legend + save ---
def plot_group_small_multiples(group_name, max_cols=5, save=True):
    sub = panel_cy[panel_cy["group_timing"] == group_name].copy()
    countries = sorted(sub["partner_id"].unique())
    n = len(countries)

    if n == 0:
        print("No countries in group:", group_name)
        return

    cols = min(max_cols, n)
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.4), sharex=True)
    axes = np.array(axes).reshape(-1)

    for ax, c in zip(axes, countries):
        d = sub[sub["partner_id"] == c].sort_values("year")

        # shares (left axis)
        ax.plot(d["year"], d["exp_pct"], linewidth=1, linestyle="-")
        ax.plot(d["year"], d["imp_pct"], linewidth=1, linestyle="--")
        ax.set_title(c, fontsize=10)
        ax.tick_params(axis="x", labelrotation=45)
        ax.set_ylim(bottom=0)

        # intensity (right axis) – up to 2023 (2024 is NaN)
        ax2 = ax.twinx()
        ax2.plot(d["year"], d["intensity"], linewidth=1, linestyle=":")
        ax2.set_ylim(0, 6)

    # disable empty subplots
    for j in range(len(countries), len(axes)):
        axes[j].axis("off")

    # suptitle
    fig.suptitle(
        f"{group_name}: country-level export/import shares and sanctions intensity",
        y=1.02,
        fontsize=12
    )

    # legend
    legend_handles = [
        Line2D([0], [0], color="C0", lw=1, linestyle="-",  label="Export share (%)"),
        Line2D([0], [0], color="C1", lw=1, linestyle="--", label="Import share (%)"),
        Line2D([0], [0], color="C0", lw=1, linestyle=":",  label="Sanctions intensity (0–6)"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="upper right",
        bbox_to_anchor=(0.995, 0.995),
        frameon=True
    )

    # layout
    plt.tight_layout(rect=[0, 0, 0.97, 0.95])

    
    if save:
        safe_name = group_name.replace(" ", "_").replace("(", "").replace(")", "").replace("<=", "le").replace(">=", "ge")
        out_file = f"appendix_small_multiples_{safe_name}.png"
        out_full = os.path.join(out_path, out_file)
        fig.savefig(out_full, dpi=300, bbox_inches="tight")
        print("Saved:", out_full)

    plt.show()

# --- 4) 3 graphs: Never / Early / Late ---
plot_group_small_multiples("Never sanctioning", max_cols=5, save=True)
plot_group_small_multiples("Early sanctioners (<=2021)", max_cols=5, save=True)
plot_group_small_multiples("Late sanctioners (>=2022)", max_cols=5, save=True)

In [ ]:
# 1) Partner -> first sanction year + group (Never/Early/Late)

tmp_s = df[df["gsdb_intensity_cap"] > 0].copy()
first_year = tmp_s.groupby("partner_id")["year"].min()

partners = pd.Index(df["partner_id"].unique(), name="partner_id")
grp = pd.DataFrame({"partner_id": partners})
grp["First sanction year"] = grp["partner_id"].map(first_year)

def classify(y):
    if pd.isna(y):
        return "Never sanctioning"
    elif y <= 2021:
        return "Early sanctioners (≤2021)"
    else:
        return "Late sanctioners (≥2022)"

grp["Group"] = grp["First sanction year"].apply(classify)

# 2) Summary of intensity via partner

def agg_intensity(d, label):
    out = (d.groupby("partner_id")["gsdb_intensity_cap"]
             .agg(**{
                 f"Avg intensity ({label})": "mean",
                 f"Max intensity ({label})": "max"
             })
          )
    return out

int_1623 = agg_intensity(df[df["year"].between(2016, 2023)], "2016–2023")
int_2223 = agg_intensity(df[df["year"].between(2022, 2023)], "2022–2023")


# 3) Share summary per partner (pre vs post)

def agg_share(d, col, label):
    return d.groupby("partner_id")[col].mean().rename(label)

pre = df[df["year"].between(2016, 2021)]
post = df[df["year"].between(2022, 2023)]

exp_pre = agg_share(pre, "exp_share", "Export share pre (2016–2021), %")
exp_post = agg_share(post, "exp_share", "Export share post (2022–2023), %")
imp_pre = agg_share(pre, "imp_share", "Import share pre (2016–2021), %")
imp_post = agg_share(post, "imp_share", "Import share post (2022–2023), %")

# percentage
exp_pre *= 100; exp_post *= 100; imp_pre *= 100; imp_post *= 100

# changes pp
exp_delta = (exp_post - exp_pre).rename("Δ export share (pp)")
imp_delta = (imp_post - imp_pre).rename("Δ import share (pp)")

# -----------------------------
# 4) Final appendix table
# -----------------------------
tab = (grp.set_index("partner_id")
         .join([int_1623, int_2223, exp_pre, exp_post, exp_delta, imp_pre, imp_post, imp_delta])
         .reset_index()
      )

# formatting
num_cols = [c for c in tab.columns if c not in ["partner_id", "Group", "First sanction year"]]
for c in num_cols:
    tab[c] = tab[c].astype(float).round(2)

# sorting
tab["Post total share (2022–2023), %"] = (tab["Export share post (2022–2023), %"] +
                                         tab["Import share post (2022–2023), %"]).round(2)

tab = tab.sort_values(
    ["Group", "Post total share (2022–2023), %"],
    ascending=[True, False]
)

tab = tab[[
    "Group",
    "partner_id",
    "First sanction year",
    "Avg intensity (2022–2023)",
    "Max intensity (2022–2023)",
    "Avg intensity (2016–2023)",
    "Max intensity (2016–2023)",
    "Export share pre (2016–2021), %",
    "Export share post (2022–2023), %",
    "Δ export share (pp)",
    "Import share pre (2016–2021), %",
    "Import share post (2022–2023), %",
    "Δ import share (pp)",
    "Post total share (2022–2023), %"
]]


(tab.style
   .hide(axis="index")
   .format({
       "First sanction year": "{:.0f}",
       "Avg intensity (2022–2023)": "{:.2f}",
       "Max intensity (2022–2023)": "{:.0f}",
       "Avg intensity (2016–2023)": "{:.2f}",
       "Max intensity (2016–2023)": "{:.0f}",
       "Export share pre (2016–2021), %": "{:.2f}",
       "Export share post (2022–2023), %": "{:.2f}",
       "Δ export share (pp)": "{:.2f}",
       "Import share pre (2016–2021), %": "{:.2f}",
       "Import share post (2022–2023), %": "{:.2f}",
       "Δ import share (pp)": "{:.2f}",
       "Post total share (2022–2023), %": "{:.2f}",
   })
)
out_path = r"C:\Users\psimo\Music\Python\Paper"
os.makedirs(out_path, exist_ok=True)

xlsx_file = os.path.join(out_path, "Appendix_Table_timing_intensity_prepost.xlsx")
csv_file  = os.path.join(out_path, "Appendix_Table_timing_intensity_prepost.csv")

tab.to_excel(xlsx_file, index=False)
tab.to_csv(csv_file, index=False, encoding="utf-8-sig")

print("Saved:")
print(xlsx_file)
print(csv_file)
styled = (tab.style
   .hide(axis="index")
   .format({
       "First sanction year": "{:.0f}",
       "Avg intensity (2022–2023)": "{:.2f}",
       "Max intensity (2022–2023)": "{:.0f}",
       "Avg intensity (2016–2023)": "{:.2f}",
       "Max intensity (2016–2023)": "{:.0f}",
       "Export share pre (2016–2021), %": "{:.2f}",
       "Export share post (2022–2023), %": "{:.2f}",
       "Δ export share (pp)": "{:.2f}",
       "Import share pre (2016–2021), %": "{:.2f}",
       "Import share post (2022–2023), %": "{:.2f}",
       "Δ import share (pp)": "{:.2f}",
       "Post total share (2022–2023), %": "{:.2f}",
   })
)
display(styled)

1) Figure: “Dumbbell” (before vs after) TOP 15 partners – export (%)/import (%)
Shows who increased/decreased share after 2022.

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper"
os.makedirs(out_path, exist_ok=True)

In [ ]:
pre = (panel_strategic[panel_strategic["year"].between(2016, 2021)]
       .groupby("COUNTERPART_COUNTRY.ID")["exp_share"].mean())
post = (panel_strategic[panel_strategic["year"].between(2022, 2024)]
        .groupby("COUNTERPART_COUNTRY.ID")["exp_share"].mean())

comp = pd.concat([pre, post], axis=1)
comp.columns = ["pre", "post"]
comp = comp.fillna(0)
comp["avg"] = (comp["pre"] + comp["post"]) / 2

top = comp.sort_values("avg", ascending=False).head(15)[["pre", "post"]] * 100
top = top.sort_values("post")

y = range(len(top))
fig, ax = plt.subplots(figsize=(7, 6))
ax.hlines(y=y, xmin=top["pre"], xmax=top["post"], linewidth=1)

ax.plot(top["pre"], y, marker="o", label="2016–2021")
ax.plot(top["post"], y, marker="o", label="2022–2024 (descriptive)")

ax.set_yticks(list(y))
ax.set_yticklabels(top.index)
ax.set_xlabel("Export share (%)")
ax.set_title("Russia export partners: share reallocation (pre vs post; post includes 2024)")
ax.legend()
fig.tight_layout()


out_full = os.path.join(out_path, "fig_export_dumbbell_prepost_2022_2024.png")
fig.savefig(out_full, dpi=300, bbox_inches="tight", facecolor="white")
print("Saved:", out_full)


plt.show()

In [ ]:
pre = (panel_strategic[panel_strategic["year"].between(2016, 2021)]
       .groupby("COUNTERPART_COUNTRY.ID")["imp_share"].mean())
post = (panel_strategic[panel_strategic["year"].between(2022, 2024)]
        .groupby("COUNTERPART_COUNTRY.ID")["imp_share"].mean())

comp = pd.concat([pre, post], axis=1)
comp.columns = ["pre", "post"]
comp = comp.fillna(0)
comp["avg"] = (comp["pre"] + comp["post"]) / 2

top = comp.sort_values("avg", ascending=False).head(15)[["pre", "post"]] * 100
top = top.sort_values("post")

y = range(len(top))
fig, ax = plt.subplots(figsize=(7, 6))
ax.hlines(y=y, xmin=top["pre"], xmax=top["post"], linewidth=1)

ax.plot(top["pre"], y, marker="o", label="2016–2021")
ax.plot(top["post"], y, marker="o", label="2022–2024 (descriptive)")

ax.set_yticks(list(y))
ax.set_yticklabels(top.index)
ax.set_xlabel("Import share (%)")
ax.set_title("Russia import partners: share reallocation (pre vs post; post includes 2024)")
ax.legend()
fig.tight_layout()

out_full = os.path.join(out_path, "fig_import_dumbbell_prepost_2022_2024.png")
fig.savefig(out_full, dpi=300, bbox_inches="tight", facecolor="white")
print("Saved:", out_full)

plt.show()

3) “Strategic map” without pictures: switchers, temporary, stable

In [ ]:
def make_partner_col(df_in, partner_col_name="Partner (ISO3)"):
    df_out = df_in.copy()
    # if the partner is in the index (most often)
    if df_out.index.name in ["partner_id", "COUNTERPART_COUNTRY.ID", None]:
        # if index is a real partner list (e.g. CHN, DEU...), reset
        if df_out.index.dtype == "object":
            df_out = df_out.reset_index()
            df_out = df_out.rename(columns={"index": partner_col_name, "partner_id": partner_col_name,
                                            "COUNTERPART_COUNTRY.ID": partner_col_name})
            return df_out
    # if the partner is already in the column
    if "partner_id" in df_out.columns:
        df_out = df_out.rename(columns={"partner_id": partner_col_name})
    if "COUNTERPART_COUNTRY.ID" in df_out.columns:
        df_out = df_out.rename(columns={"COUNTERPART_COUNTRY.ID": partner_col_name})
    return df_out

In [ ]:
# Create tab_pct
pre_years = range(2016, 2022)   # 2016–2021
post_years = range(2022, 2024)  # 2022–2023

if "partner_id" not in df.columns and "COUNTERPART_COUNTRY.ID" in df.columns:
    df = df.rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"})

exp_pre  = df[df["year"].isin(pre_years)].groupby("partner_id")["exp_share"].mean()
exp_post = df[df["year"].isin(post_years)].groupby("partner_id")["exp_share"].mean()
imp_pre  = df[df["year"].isin(pre_years)].groupby("partner_id")["imp_share"].mean()
imp_post = df[df["year"].isin(post_years)].groupby("partner_id")["imp_share"].mean()

all_ids = sorted(df["partner_id"].unique())
tab = pd.DataFrame(index=all_ids)
tab["exp_pre"]  = exp_pre.reindex(all_ids).fillna(0)
tab["exp_post"] = exp_post.reindex(all_ids).fillna(0)
tab["imp_pre"]  = imp_pre.reindex(all_ids).fillna(0)
tab["imp_post"] = imp_post.reindex(all_ids).fillna(0)

tab_pct = tab * 100

tab_pct["exp_change_pp"] = tab_pct["exp_post"] - tab_pct["exp_pre"]
tab_pct["imp_change_pp"] = tab_pct["imp_post"] - tab_pct["imp_pre"]
tab_pct["total_post"] = tab_pct["exp_post"] + tab_pct["imp_post"]

tab_pct["strategic"] = ((tab_pct["exp_post"] >= 5) | (tab_pct["imp_post"] >= 5) | (tab_pct["total_post"] >= 8))

tab_pct["role_post"] = np.where(tab_pct["exp_post"] - tab_pct["imp_post"] >= 2, "export-leaning",
                         np.where(tab_pct["imp_post"] - tab_pct["exp_post"] >= 2, "import-leaning", "balanced"))

print("tab_pct built:", tab_pct.shape)
display(tab_pct.head())

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper" 
os.makedirs(out_path, exist_ok=True)

# ---- 1) TOP export increases (pp) ----
top_exp = (tab_pct.sort_values("exp_change_pp", ascending=False)
           [["exp_pre","exp_post","exp_change_pp","role_post","strategic"]]
           .head(10)
           .copy())

top_exp = top_exp.reset_index().rename(columns={"index":"Partner (ISO3)"})
top_exp = top_exp.rename(columns={
    "exp_pre": "Export share pre (2016–2021), %",
    "exp_post": "Export share post (2022–2023), %",
    "exp_change_pp": "Δ export share (pp)",
    "role_post": "Role post-2022",
    "strategic": "Strategic (rule-based)"
})
for c in ["Export share pre (2016–2021), %","Export share post (2022–2023), %","Δ export share (pp)"]:
    top_exp[c] = top_exp[c].astype(float).round(2)

# without index
display(top_exp.style.hide(axis="index").format({
    "Export share pre (2016–2021), %":"{:.2f}",
    "Export share post (2022–2023), %":"{:.2f}",
    "Δ export share (pp)":"{:.2f}",
}))


top_exp.to_csv(os.path.join(out_path, "table_top_export_increases.csv"), index=False)
top_exp.to_excel(os.path.join(out_path, "table_top_export_increases.xlsx"), index=False)


# ---- 2) TOP import increases (pp) ----
top_imp = (tab_pct.sort_values("imp_change_pp", ascending=False)
           [["imp_pre","imp_post","imp_change_pp","role_post","strategic"]]
           .head(10)
           .copy())

top_imp = top_imp.reset_index().rename(columns={"index":"Partner (ISO3)"})
top_imp = top_imp.rename(columns={
    "imp_pre": "Import share pre (2016–2021), %",
    "imp_post": "Import share post (2022–2023), %",
    "imp_change_pp": "Δ import share (pp)",
    "role_post": "Role post-2022",
    "strategic": "Strategic (rule-based)"
})
for c in ["Import share pre (2016–2021), %","Import share post (2022–2023), %","Δ import share (pp)"]:
    top_imp[c] = top_imp[c].astype(float).round(2)

display(top_imp.style.hide(axis="index").format({
    "Import share pre (2016–2021), %":"{:.2f}",
    "Import share post (2022–2023), %":"{:.2f}",
    "Δ import share (pp)":"{:.2f}",
}))

top_imp.to_csv(os.path.join(out_path, "table_top_import_increases.csv"), index=False)
top_imp.to_excel(os.path.join(out_path, "table_top_import_increases.xlsx"), index=False)


# ---- 3) Strategic partners table ----
strategic_tbl = (tab_pct[tab_pct["strategic"]]
                 .sort_values("total_post", ascending=False)
                 [["exp_post","imp_post","total_post","role_post"]]
                 .copy())

strategic_tbl = strategic_tbl.reset_index().rename(columns={"index":"Partner (ISO3)"})
strategic_tbl = strategic_tbl.rename(columns={
    "exp_post": "Export share post (2022–2023), %",
    "imp_post": "Import share post (2022–2023), %",
    "total_post": "Total post share (exp+imp), %",
    "role_post": "Role post-2022"
})
for c in ["Export share post (2022–2023), %","Import share post (2022–2023), %","Total post share (exp+imp), %"]:
    strategic_tbl[c] = strategic_tbl[c].astype(float).round(2)

display(strategic_tbl.style.hide(axis="index").format({
    "Export share post (2022–2023), %":"{:.2f}",
    "Import share post (2022–2023), %":"{:.2f}",
    "Total post share (exp+imp), %":"{:.2f}",
}))

strategic_tbl.to_csv(os.path.join(out_path, "table_strategic_partners.csv"), index=False)
strategic_tbl.to_excel(os.path.join(out_path, "table_strategic_partners.xlsx"), index=False)

print("Saved 3 tables to:", out_path)

4) 2-panel scatter: with CHN and without CHN, post = 2022–2024 (TOP8 labels)

In [ ]:
post = panel_strategic[panel_strategic["year"].between(2022, 2024)].copy()

pt = (post.groupby("COUNTERPART_COUNTRY.ID", as_index=False)[["exp_share","imp_share"]].mean())
pt.rename(columns={"COUNTERPART_COUNTRY.ID":"partner_id"}, inplace=True)

pt["exp_post_pct"] = pt["exp_share"] * 100
pt["imp_post_pct"] = pt["imp_share"] * 100
pt["total_post_pct"] = pt["exp_post_pct"] + pt["imp_post_pct"]

top8 = pt.sort_values("total_post_pct", ascending=False).head(8)

pt_nochn = pt[pt["partner_id"] != "CHN"].copy()
top8_nochn = pt_nochn.sort_values("total_post_pct", ascending=False).head(8)

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=False, sharey=False)

# LEFT: with China
axes[0].scatter(pt["exp_post_pct"], pt["imp_post_pct"])
axes[0].set_title("Post-2022 trade composition (with China)")
axes[0].set_xlabel("Avg export share (2022–2024), %")
axes[0].set_ylabel("Avg import share (2022–2024), %")
for _, r in top8.iterrows():
    axes[0].text(r["exp_post_pct"], r["imp_post_pct"], r["partner_id"], fontsize=9)

# RIGHT: excluding China (zoom)
axes[1].scatter(pt_nochn["exp_post_pct"], pt_nochn["imp_post_pct"])
axes[1].set_title("Post-2022 trade composition (excluding China)")
axes[1].set_xlabel("Avg export share (2022–2024), %")
axes[1].set_ylabel("Avg import share (2022–2024), %")
axes[1].set_xlim(0, 14)
axes[1].set_ylim(0, 8)
for _, r in top8_nochn.iterrows():
    axes[1].text(r["exp_post_pct"], r["imp_post_pct"], r["partner_id"], fontsize=9)

plt.tight_layout()
out_file = "fig_scatter_post2022_with_without_CHN.png"
out_full = os.path.join(out_path, out_file)

plt.tight_layout()
plt.savefig(out_full, dpi=300, bbox_inches="tight")
print("Saved:", out_full)

plt.show()

In [ ]:
# 1) Post period (trade-only): 2022–2024
post = panel_strategic[panel_strategic["year"].between(2022, 2024)].copy()

# 2) Country averages (export/import shares)
pt = (post.groupby("COUNTERPART_COUNTRY.ID", as_index=False)[["exp_share", "imp_share"]]
          .mean()
     )

pt = pt.rename(columns={"COUNTERPART_COUNTRY.ID": "Partner (ISO3)"})

# 3) Convert shares to percent + total
pt["Export share (2022–2024), %"] = (pt["exp_share"] * 100).round(2)
pt["Import share (2022–2024), %"] = (pt["imp_share"] * 100).round(2)
pt["Total share (exp+imp), %"] = (pt["Export share (2022–2024), %"] + pt["Import share (2022–2024), %"]).round(2)

# 4) Top8 labels
top8_with_chn = set(pt.sort_values("Total share (exp+imp), %", ascending=False).head(8)["Partner (ISO3)"])

pt_nochn = pt[pt["Partner (ISO3)"] != "CHN"].copy()
top8_excl_chn = set(pt_nochn.sort_values("Total share (exp+imp), %", ascending=False).head(8)["Partner (ISO3)"])

pt["Top8 label (with CHN)"] = pt["Partner (ISO3)"].isin(top8_with_chn)
pt["Top8 label (excl. CHN)"] = pt["Partner (ISO3)"].isin(top8_excl_chn)

# 5) Keep only clean columns (no raw exp_share/imp_share)
table_scatter = pt[[
    "Partner (ISO3)",
    "Export share (2022–2024), %",
    "Import share (2022–2024), %",
    "Total share (exp+imp), %",
    "Top8 label (with CHN)",
    "Top8 label (excl. CHN)",
]].sort_values("Total share (exp+imp), %", ascending=False).reset_index(drop=True)

# 6) Display WITHOUT row numbering (index)
display(
    table_scatter.style
        .hide(axis="index")
        .format({
            "Export share (2022–2024), %": "{:.2f}",
            "Import share (2022–2024), %": "{:.2f}",
            "Total share (exp+imp), %": "{:.2f}",
        })
)

table_scatter.to_excel("scatter_table_2022_2024.xlsx", index=False)
table_scatter.to_csv("scatter_table_2022_2024.csv", index=False)

“Stacked bar” TOP8: total_post broken down into export vs import (very clear)

In [ ]:
# --- TOP8 stacked bar: post-2022 (incl. 2024) export vs import decomposition ---

# 1) post period
post = panel_strategic[panel_strategic["year"].between(2022, 2024)].copy()

# 2) partner averages in shares (0-1), then convert to %
pt = (post.groupby("COUNTERPART_COUNTRY.ID", as_index=False)[["exp_share", "imp_share"]].mean()
      .rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"}))

pt["Export share (2022–2024), %"] = pt["exp_share"] * 100
pt["Import share (2022–2024), %"] = pt["imp_share"] * 100
pt["Total share (export+import), %"] = pt["Export share (2022–2024), %"] + pt["Import share (2022–2024), %"]

# 3) TOP8 by total share, sorted for nicer barh order (small->large)
top8 = pt.sort_values("Total share (export+import), %", ascending=False).head(8).copy()
top8_plot = top8.sort_values("Total share (export+import), %", ascending=True).copy()

# 4) plot
plt.figure(figsize=(7, 6))
plt.barh(
    top8_plot["partner_id"],
    top8_plot["Export share (2022–2024), %"],
    label="Export share (%, 2022–2024)"
)
plt.barh(
    top8_plot["partner_id"],
    top8_plot["Import share (2022–2024), %"],
    left=top8_plot["Export share (2022–2024), %"],
    label="Import share (%, 2022–2024)"
)
plt.xlabel("Share (%)")
plt.title("Top partners post-2022 (incl. 2024): export vs import decomposition")
plt.legend()
plt.tight_layout()
out_path = r"C:\Users\psimo\Music\Python\Paper"
out_file = "fig_top8_stackedbar_post2022_incl2024.png"
out_full = os.path.join(out_path, out_file)
plt.savefig(out_full, dpi=300, bbox_inches="tight")
print("Saved:", out_full)

plt.show()

# 5) table for appendix: nice labels, 2 decimals shown, NO index
top8_table = top8_plot[[
    "partner_id",
    "Export share (2022–2024), %",
    "Import share (2022–2024), %",
    "Total share (export+import), %"
]].copy()

top8_table = top8_table.rename(columns={"partner_id": "Partner (ISO3)"})

display(
    top8_table.style
    .hide(axis="index")
    .format({
        "Export share (2022–2024), %": "{:.2f}",
        "Import share (2022–2024), %": "{:.2f}",
        "Total share (export+import), %": "{:.2f}",
    })
)

!!! Trade descriptives are shown through 2024, while econometric estimates are restricted to 2016–2023 due to the availability of sanctions data.

ROBUSTNESS: GRAVITY WITH PPML AND FE - EXPORT

In [ ]:
# 0) Quick diagnostics: what DFs are and what columns are in them

def quick_check(df_obj, name, n=10):
    print(f"\n=== {name} ===")
    try:
        print("shape:", df_obj.shape)
        print("columns:", df_obj.columns.tolist())
        display(df_obj.head(n))
    except Exception as e:
        print("ERROR:", e)

quick_check(panel_strategic, "panel_strategic (trade-only, 2016–2024)")
quick_check(gsdb_ru,         "gsdb_ru (Russia-only GSDB episodes)")
quick_check(gsdb_ready,      "gsdb_ready (partner-year intensity cap)")
quick_check(df,              "df (merged panel for estimation, 2016–2023)")

# 1) Find a candidate for "trade level" columns (export/import levels, USD, etc.)

def find_trade_level_cols(df_obj):
    cols = df_obj.columns.tolist()
    pat = re.compile(r"(export|exports|import|imports|trade|value|usd|amount)", re.IGNORECASE)
    return [c for c in cols if pat.search(c)]

print("\nCandidate trade-level columns in panel_strategic:", find_trade_level_cols(panel_strategic))
print("Candidate trade-level columns in df:", find_trade_level_cols(df))

# 2) Quick sanity check: does df have a partner_id? if not - let's create one temporarily

if "partner_id" not in df.columns and "COUNTERPART_COUNTRY.ID" in df.columns:
    df = df.rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"})

# 3) Check if df has at least one "level" (not share) variable and how many zeros
level_cols_df = [c for c in find_trade_level_cols(df) if c not in ["exp_share","imp_share"]]
print("\nTrade-level-ish cols in df (excluding shares):", level_cols_df)

if level_cols_df:
    for c in level_cols_df[:6]:
        s = df[c]
        # if number
        if hasattr(s, "dtype"):
            try:
                print(f"\n{c}: missing={s.isna().mean():.3f}, zeros={(s==0).mean():.3f}, min={s.min()}, max={s.max()}")
            except Exception:
                pass

# 4) Check if panel_strategic has levels
level_cols_ps = [c for c in find_trade_level_cols(panel_strategic) if c not in ["exp_share","imp_share"]]
print("\nTrade-level-ish cols in panel_strategic (excluding shares):", level_cols_ps)

if level_cols_ps:
    for c in level_cols_ps[:6]:
        s = panel_strategic[c]
        try:
            print(f"\n{c}: missing={s.isna().mean():.3f}, zeros={(s==0).mean():.3f}, min={s.min()}, max={s.max()}")
        except Exception:
            pass

In [ ]:
# 1) Check if there is what is needed for export (levels)
req_cols = ["COUNTERPART_COUNTRY.ID", "year", "Exports of goods", "Imports of goods"]
missing = [c for c in req_cols if c not in panel_strategic.columns]
print("Missing in panel_strategic:", missing)

# 2) Minimal preview
display(panel_strategic[req_cols + ["exp_share","imp_share","trade_balance"]].head(10))

#3) Make a "clean" copy with safe names
panel_g = panel_strategic.rename(columns={
    "COUNTERPART_COUNTRY.ID": "partner_id",
    "Exports of goods": "x_goods",   # export level (USD)
    "Imports of goods": "m_goods",   # import level (USD)
})

# 4) Sanity check
print(panel_g[["partner_id","year","x_goods","m_goods"]].dtypes)
print("Zero share in x_goods:", (panel_g["x_goods"]==0).mean())
print("Zero share in m_goods:", (panel_g["m_goods"]==0).mean())
display(panel_g[["partner_id","year","x_goods","m_goods"]].head(10))

1. Estimation panel (2016-2023) merge with sanctions

In [ ]:
# 1) leave 2016–2023 (because sanctions up to 2023)
panel_g_1623 = panel_g[panel_g["year"].between(2016, 2023)].copy()

gsdb_ready = gsdb_ready.rename(columns={"COUNTERPART_COUNTRY.ID": "partner_id"})

gsdb_1623 = gsdb_ready[gsdb_ready["year"].between(2016, 2023)].copy()

df_est = panel_g_1623.merge(
    gsdb_1623[["partner_id", "year", "gsdb_intensity_cap"]],
    on=["partner_id", "year"],
    how="left"
)

df_est["gsdb_intensity_cap"] = df_est["gsdb_intensity_cap"].fillna(0).astype(int)

print("Rows:", df_est.shape[0], "Partners:", df_est["partner_id"].nunique())
print("Non-zero sanction share:", (df_est["gsdb_intensity_cap"] > 0).mean())

display(df_est[["partner_id","year","x_goods","m_goods","gsdb_intensity_cap"]].head(12))

2. Minimal sanity check before PPLM (for export)

In [ ]:
print("Any negative x_goods?", (df_est["x_goods"] < 0).any())
print("Any missing x_goods?", df_est["x_goods"].isna().mean())

print("Zero share in x_goods:", (df_est["x_goods"] == 0).mean())

by_partner_var = df_est.groupby("partner_id")["gsdb_intensity_cap"].nunique()
print("Partners with NO within variation (unique==1):", (by_partner_var == 1).sum(), "/", len(by_partner_var))

In [ ]:
req = ["partner_id", "year", "gsdb_intensity_cap", "x_goods", "m_goods"]
missing = [c for c in req if c not in df_est.columns]
print("Missing in df_est:", missing)
print(df_est[req].head())
print("Partners:", df_est["partner_id"].nunique(), "Years:", df_est["year"].nunique())

PPML export + UKR exluded

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper\\"
os.makedirs(out_path, exist_ok=True)

def save_full_model_output(model, name, out_path):
    txt_path = os.path.join(out_path, f"{name}_FULL_SUMMARY.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(model.summary().as_text())

    html_path = os.path.join(out_path, f"{name}_FULL_SUMMARY.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(model.summary().as_html())

    coef_path = os.path.join(out_path, f"{name}_COEFS.csv")
    model.summary2().tables[1].to_csv(coef_path, encoding="utf-8-sig")

    print("Saved:", txt_path)
    print("Saved:", html_path)
    print("Saved:", coef_path)

# A) PPML EXPORT baseline

ppml_exp = smf.glm(
    formula="x_goods ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df_est,
    family=sm.families.Poisson()
).fit(cov_type="cluster", cov_kwds={"groups": df_est["partner_id"]})

print("\n=== PPML EXPORT (baseline) ===")
print(ppml_exp.summary().tables[1])

beta = ppml_exp.params.get("gsdb_intensity_cap", np.nan)
print("Semi-elasticity (% effect per +1 intensity):", 100*(np.exp(beta)-1))
print("Log-likelihood:", round(ppml_exp.llf, 4))
print("N:", int(ppml_exp.nobs))

save_full_model_output(ppml_exp, "PPML_export_baseline", out_path)

# B) PPML EXPORT robustness: exclude UKR

df_est_no_ukr = df_est[df_est["partner_id"] != "UKR"].copy()

ppml_exp_no_ukr = smf.glm(
    formula="x_goods ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df_est_no_ukr,
    family=sm.families.Poisson()
).fit(cov_type="cluster", cov_kwds={"groups": df_est_no_ukr["partner_id"]})

print("\n=== PPML EXPORT (exclude UKR) ===")
print(ppml_exp_no_ukr.summary().tables[1])

beta_no = ppml_exp_no_ukr.params.get("gsdb_intensity_cap", np.nan)
print("Semi-elasticity (% effect per +1 intensity):", 100*(np.exp(beta_no)-1))
print("Log-likelihood:", round(ppml_exp_no_ukr.llf, 4))
print("N:", int(ppml_exp_no_ukr.nobs))

save_full_model_output(ppml_exp_no_ukr, "PPML_export_noUKR", out_path)

PPML import + UKR excluded

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper\\"
os.makedirs(out_path, exist_ok=True)

def save_full_model_output(model, name, out_path):
    txt_path = os.path.join(out_path, f"{name}_FULL_SUMMARY.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(model.summary().as_text())

    html_path = os.path.join(out_path, f"{name}_FULL_SUMMARY.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(model.summary().as_html())

    coef_path = os.path.join(out_path, f"{name}_COEFS.csv")
    model.summary2().tables[1].to_csv(coef_path, encoding="utf-8-sig")

    print("Saved:", txt_path)
    print("Saved:", html_path)
    print("Saved:", coef_path)

# A) PPML IMPORT baseline

ppml_imp = smf.glm(
    formula="m_goods ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df_est,
    family=sm.families.Poisson()
).fit(cov_type="cluster", cov_kwds={"groups": df_est["partner_id"]})

print("\n=== PPML IMPORT (baseline) ===")
print(ppml_imp.summary().tables[1])

beta = ppml_imp.params.get("gsdb_intensity_cap", np.nan)
print("Semi-elasticity (% effect per +1 intensity):", 100*(np.exp(beta)-1))
print("Log-likelihood:", round(ppml_imp.llf, 4))
print("N:", int(ppml_imp.nobs))

save_full_model_output(ppml_imp, "PPML_import_baseline", out_path)

# B) PPML IMPORT robustness: exclude UKR

df_est_no_ukr = df_est[df_est["partner_id"] != "UKR"].copy()

ppml_imp_no_ukr = smf.glm(
    formula="m_goods ~ gsdb_intensity_cap + C(partner_id) + C(year)",
    data=df_est_no_ukr,
    family=sm.families.Poisson()
).fit(cov_type="cluster", cov_kwds={"groups": df_est_no_ukr["partner_id"]})

print("\n=== PPML IMPORT (exclude UKR) ===")
print(ppml_imp_no_ukr.summary().tables[1])

beta_no = ppml_imp_no_ukr.params.get("gsdb_intensity_cap", np.nan)
print("Semi-elasticity (% effect per +1 intensity):", 100*(np.exp(beta_no)-1))
print("Log-likelihood:", round(ppml_imp_no_ukr.llf, 4))
print("N:", int(ppml_imp_no_ukr.nobs))

save_full_model_output(ppml_imp_no_ukr, "PPML_import_noUKR", out_path)

1) OLS (shares) results table (4 columns: exp/imp × baseline/no UKR)

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper"
os.makedirs(out_path, exist_ok=True)

var = "gsdb_intensity_cap"
dec = 3  

def extract_row(res, varname=var):
    b = float(res.params[varname])
    se = float(res.bse[varname])
    p = float(res.pvalues[varname])
    ci_low, ci_high = [float(x) for x in res.conf_int().loc[varname].tolist()]
    n = int(getattr(res, "nobs", np.nan))
    r2 = float(getattr(res, "rsquared", np.nan))
    r2_adj = float(getattr(res, "rsquared_adj", np.nan))
    
    return {
        "Coef": b,
        "Std. error": se,
        "p-value": p,
        "CI 95% (low)": ci_low,
        "CI 95% (high)": ci_high,
        "R²": r2,
        "Adj. R²": r2_adj,
        "N": n
    }

ols_specs = [
    ("Export share — baseline", m_exp),
    ("Export share — excl. UKR", m_exp_no_ukr),
    ("Import share — baseline", m_imp),
    ("Import share — excl. UKR", m_imp_no_ukr),
]

rows = []
for name, model in ols_specs:
    r = extract_row(model, var)
    r["Model"] = name
    r["Estimator"] = "OLS"
    r["Partner FE"] = "Yes"
    r["Year FE"] = "Yes"
    r["SE type"] = "Clustered by partner"
    rows.append(r)

ols_table = pd.DataFrame(rows)

ols_table = ols_table[[
    "Model", "Estimator",
    "Coef", "Std. error", "p-value", "CI 95% (low)", "CI 95% (high)",
    "R²", "Adj. R²", "N", "Partner FE", "Year FE", "SE type"
]].copy()

for c in ["Coef", "Std. error", "p-value", "CI 95% (low)", "CI 95% (high)", "R²", "Adj. R²"]:
    ols_table[c] = ols_table[c].round(dec)

# failai
csv_path = os.path.join(out_path, "Table_OLS_shares_gsdb_intensity.csv")
html_path = os.path.join(out_path, "Table_OLS_shares_gsdb_intensity.html")

ols_table.to_csv(csv_path, index=False, encoding="utf-8-sig")
ols_table.style.hide(axis="index").to_html(html_path)

print("Saved OLS table to:")
print(csv_path)
print(html_path)

display(ols_table)

2) PPML (levels) results table (4 columns: exp/imp × baseline/no UKR)

In [ ]:
out_path = r"C:\Users\psimo\Music\Python\Paper\\"

def ppml_row(model, label, var="gsdb_intensity_cap"):
    # coef + se + pvalue
    coef = float(model.params[var])
    se = float(model.bse[var])
    p = float(model.pvalues[var])

    # 95% CI coef
    ci_low = coef - 1.96 * se
    ci_high = coef + 1.96 * se

    # Semi-elasticity 
    semi = 100.0 * (np.exp(coef) - 1.0)
    semi_low = 100.0 * (np.exp(ci_low) - 1.0)
    semi_high = 100.0 * (np.exp(ci_high) - 1.0)

    # N (observations) + log-likelihood
    n = int(model.nobs)
    ll = float(getattr(model, "llf", np.nan))

    return {
        "Model": label,
        "Estimator": "PPML",
        "Coef (β)": coef,
        "SE": se,
        "p-value": p,
        "CI 95% (β) low": ci_low,
        "CI 95% (β) high": ci_high,
        "Semi-elasticity (% per +1)": semi,
        "CI 95% semi low": semi_low,
        "CI 95% semi high": semi_high,
        "Log-likelihood": ll,
        "N": n,
        "Partner FE": "Yes",
        "Year FE": "Yes",
        "SE type": "Clustered by partner",
    }


ppml_table = pd.DataFrame([
    ppml_row(ppml_exp, "Exports — baseline"),
    ppml_row(ppml_exp_no_ukr, "Exports — excl. UKR"),
    ppml_row(ppml_imp, "Imports — baseline"),
    ppml_row(ppml_imp_no_ukr, "Imports — excl. UKR"),
])


round_cols_3 = ["Coef (β)", "SE", "CI 95% (β) low", "CI 95% (β) high", "Log-likelihood"]
round_cols_2 = ["Semi-elasticity (% per +1)", "CI 95% semi low", "CI 95% semi high"]

ppml_table[round_cols_3] = ppml_table[round_cols_3].round(3)
ppml_table[round_cols_2] = ppml_table[round_cols_2].round(2)
ppml_table["p-value"] = ppml_table["p-value"].round(3)

ppml_table = ppml_table[[
    "Model", "Estimator",
    "Coef (β)", "SE", "p-value",
    "CI 95% (β) low", "CI 95% (β) high",
    "Semi-elasticity (% per +1)",
    "CI 95% semi low", "CI 95% semi high",
    "Log-likelihood", "N",
    "Partner FE", "Year FE", "SE type"
]].copy()


csv_path = os.path.join(out_path, "Table_PPML_trade_gsdb_intensity.csv")
html_path = os.path.join(out_path, "Table_PPML_trade_gsdb_intensity.html")

ppml_table.to_csv(csv_path, index=False, encoding="utf-8-sig")
ppml_table.style.hide(axis="index").to_html(html_path)

print("Saved PPML table to:")
print(csv_path)
print(html_path)

display(ppml_table)